# Qwen2.5-14B Debate Card Span Fine-tuning
This notebook builds a clean aligned span dataset from `input_full.json` / `output_full.json`, formats it for Qwen chat fine-tuning with Unsloth, and trains a QLoRA model.

In [ ]:
# =========================
# INSTALL
# =========================
!pip -q install unsloth datasets trl accelerate bitsandbytes peft sentencepiece


In [ ]:
# =========================
# IMPORTS
# =========================
import os, re, json, random, time
from collections import defaultdict
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_FULL_PATH  = "/content/drive/MyDrive/cards_output_processed/input_full.json"
OUTPUT_FULL_PATH = "/content/drive/MyDrive/cards_output_processed/output_full.json"
OUTPUT_DIR       = "/content/drive/MyDrive/qwen14b_span_model"

MODEL_NAME       = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH   = 8192
LOAD_IN_4BIT     = True

RANDOM_SEED = 42
TEST_SIZE = 0.05

PER_DEVICE_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)


In [ ]:
# =========================
# LOAD INPUT / OUTPUT DATA
# =========================
with open(INPUT_FULL_PATH, "r", encoding="utf-8") as f:
    input_data = json.load(f)

with open(OUTPUT_FULL_PATH, "r", encoding="utf-8") as f:
    output_data = json.load(f)

print("Input examples:", len(input_data))
print("Output examples:", len(output_data))
assert len(input_data) == len(output_data), "input_full and output_full lengths differ"

for i, (inp, out) in enumerate(zip(input_data, output_data)):
    assert inp["tag"] == out["tag"], f"Tag mismatch at {i}"
    assert inp["cite"] == out["cite"], f"Cite mismatch at {i}"

print("Alignment check passed.")
print("\nSample input example:")
print(json.dumps(input_data[0], indent=2, ensure_ascii=False)[:1200])
print("\nSample output example:")
print(json.dumps(output_data[0], indent=2, ensure_ascii=False)[:1200])


In [ ]:
# =========================
# FAST SINGLE-PASS TAG PARSER
# =========================
VALID_TAGS = {"ul", "h", "e"}

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = s.replace("\xa0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n+", "\n", s)
    s = "\n".join(line.strip() for line in s.split("\n")).strip()
    return s

def extract_spans_from_tagged_text(tagged_text: str):
    plain_chars = []
    spans = []
    open_starts = defaultdict(list)
    i = 0
    n = len(tagged_text)
    while i < n:
        ch = tagged_text[i]
        if ch == "<":
            close_idx = tagged_text.find(">", i)
            if close_idx == -1:
                plain_chars.append(tagged_text[i])
                i += 1
                continue
            raw_tag = tagged_text[i + 1:close_idx].strip()
            if raw_tag.startswith("/"):
                tag_name = raw_tag[1:].strip()
                if tag_name in VALID_TAGS and open_starts[tag_name]:
                    start = open_starts[tag_name].pop()
                    end = len(plain_chars)
                    if end > start:
                        spans.append((start, end, tag_name))
                i = close_idx + 1
                continue
            tag_name = raw_tag
            if tag_name in VALID_TAGS:
                open_starts[tag_name].append(len(plain_chars))
                i = close_idx + 1
                continue
            plain_chars.append(tagged_text[i])
            i += 1
            continue
        plain_chars.append(ch)
        i += 1
    end_pos = len(plain_chars)
    for tag_name, starts in open_starts.items():
        while starts:
            start = starts.pop()
            if end_pos > start:
                spans.append((start, end_pos, tag_name))
    spans.sort(key=lambda x: (x[0], -x[1], x[2]))
    return "".join(plain_chars), spans


In [ ]:
# =========================
# QUICK PARSER SANITY CHECK
# =========================
start = time.time()
for out in output_data[:1000]:
    extract_spans_from_tagged_text(out["text_body"])
print("Time for 1000 examples:", round(time.time() - start, 3), "seconds")

mismatch_count = 0
examples_with_spans = 0
tag_counts = {"ul": 0, "h": 0, "e": 0}

for inp, out in zip(input_data, output_data):
    inp_text = normalize_text(inp["text_body"])
    plain_text, spans = extract_spans_from_tagged_text(normalize_text(out["text_body"]))
    if plain_text != inp_text:
        mismatch_count += 1
    if spans:
        examples_with_spans += 1
        for _, _, t in spans:
            tag_counts[t] += 1

print("Plain-text mismatches:", mismatch_count)
print("Examples with at least one span:", examples_with_spans)
print("Tag span counts:", tag_counts)


In [ ]:
# =========================
# FILTER TO EXACTLY-ALIGNED EXAMPLES
# =========================
clean_examples = []
dropped_examples = []

for idx, (inp, out) in enumerate(zip(input_data, output_data)):
    inp_text = normalize_text(inp["text_body"])
    plain_text, spans = extract_spans_from_tagged_text(normalize_text(out["text_body"]))
    if plain_text == inp_text:
        clean_examples.append({
            "tag": inp.get("tag", ""),
            "text_body": inp_text,
            "spans": spans,
            "output_text_body": out["text_body"],
            "source_index": idx,
        })
    else:
        dropped_examples.append({
            "tag": inp.get("tag", ""),
            "input_text_body": inp["text_body"],
            "normalized_input_text_body": inp_text,
            "reconstructed_plain_text": plain_text,
            "output_text_body": out["text_body"],
            "source_index": idx,
        })

total_examples = len(input_data)
kept_examples = len(clean_examples)
dropped_count = len(dropped_examples)

print(f"Total examples: {total_examples}")
print(f"Kept aligned examples: {kept_examples}")
print(f"Dropped mismatched examples: {dropped_count}")
print(f"Kept percentage: {kept_examples / total_examples * 100:.2f}%")


In [ ]:
# =========================
# BUILD TRAINING RECORDS
# =========================
def serialize_spans(spans):
    return json.dumps([[s, e, t] for s, e, t in spans], ensure_ascii=False)

train_records = []
for ex in clean_examples:
    prompt = (
        f"Tag: {ex['tag']}\n"
        f"Text:\n{ex['text_body']}\n\n"
        f"Output spans:"
    )
    response = serialize_spans(ex["spans"])
    train_records.append({
        "prompt": prompt,
        "response": response,
        "tag": ex["tag"],
        "text_body": ex["text_body"],
        "spans": ex["spans"],
    })

print("Training records:", len(train_records))
print("\nSample prompt:\n")
print(train_records[0]["prompt"][:2000])
print("\nSample response:\n")
print(train_records[0]["response"][:1000])


In [ ]:
# =========================
# TRAIN / DEV SPLIT
# =========================
idxs = list(range(len(train_records)))
random.Random(RANDOM_SEED).shuffle(idxs)
dev_size = max(1, int(len(idxs) * TEST_SIZE))
train_split = [train_records[i] for i in idxs[dev_size:]]
dev_split   = [train_records[i] for i in idxs[:dev_size]]

print("Train examples:", len(train_split))
print("Dev examples:", len(dev_split))


In [ ]:
# =========================
# LOAD MODEL + TOKENIZER
# =========================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
)


In [ ]:
# =========================
# FORMAT DATASETS FOR QWEN CHAT TEMPLATE
# =========================

from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

# Keep this short for training speed.
SYSTEM_PROMPT = """Extract formatting spans from debate card text.

Debate card context:
- The input is the text body of a debate evidence card.
- Formatting marks argumentative importance.

Span types:
- "ul" = underlined text
- "h" = highlighted text
- "e" = emphasized text

Output format:
Return only a compact JSON array of triples:
[[start, end, "type"], ...]

Rules:
- Offsets refer to the exact plain text shown by the user.
- start is inclusive, end is exclusive.
- Count spaces, punctuation, and newline characters.
- Do not rewrite or explain anything.
- Do not output markdown or code fences.
- If there are no spans, return [].
"""

def to_messages(prompt, response):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
    }

train_chat = Dataset.from_list(
    [to_messages(x["prompt"], x["response"]) for x in train_split]
)
dev_chat = Dataset.from_list(
    [to_messages(x["prompt"], x["response"]) for x in dev_split]
)

def format_chat_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_chat.map(
    format_chat_example,
    remove_columns=train_chat.column_names,
    desc="Formatting train chat examples",
)

dev_ds = dev_chat.map(
    format_chat_example,
    remove_columns=dev_chat.column_names,
    desc="Formatting dev chat examples",
)

print(train_ds)
print(dev_ds)

print("\nSystem prompt length:", len(SYSTEM_PROMPT))
print("\nSample formatted text:\n")
print(train_ds[0]["text"][:2500])

In [ ]:
# =========================
# SANITY CHECK FOR CHAT FORMATTING
# =========================
sample_text = train_ds[0]["text"]

print("Contains assistant marker:", "<|im_start|>assistant" in sample_text)
print("Contains user marker:", "<|im_start|>user" in sample_text)
print("Contains system marker:", "<|im_start|>system" in sample_text)

print("\nLast 1200 chars of sample:\n")
print(sample_text[-1200:])


In [ ]:
# =========================
# TRAINER SETUP
# =========================
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        eval_steps=EVAL_STEPS,
        save_steps=SAVE_STEPS,
        eval_strategy="steps",
        save_strategy="steps",
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        seed=RANDOM_SEED,
        bf16=True,
        report_to="none",
        packing=True,
    ),
)

print(trainer)


In [ ]:
# =========================
# TRAIN
# =========================
train_result = trainer.train()
print(train_result)


In [ ]:
# =========================
# TOKEN LENGTH ANALYSIS
# =========================

import numpy as np
from tqdm import tqdm

lengths = []

# Sample size (adjust if you want faster)
SAMPLE_SIZE = min(5000, len(train_ds))

for i in tqdm(range(SAMPLE_SIZE)):
    text = train_ds[i]["text"]
    tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
    lengths.append(len(tokens))

lengths = np.array(lengths)

print("=== TOKEN LENGTH STATS ===")
print(f"Samples analyzed: {len(lengths)}")
print(f"Min: {lengths.min()}")
print(f"Mean: {int(lengths.mean())}")
print(f"Median: {int(np.median(lengths))}")
print(f"90th percentile: {int(np.percentile(lengths, 90))}")
print(f"95th percentile: {int(np.percentile(lengths, 95))}")
print(f"99th percentile: {int(np.percentile(lengths, 99))}")
print(f"Max: {lengths.max()}")

# Check how many exceed your max length
MAX_LEN = 2048
overflow = (lengths > MAX_LEN).sum()

print(f"\n=== OVERFLOW ANALYSIS (>{MAX_LEN}) ===")
print(f"Count: {overflow}")
print(f"Percent: {overflow / len(lengths) * 100:.2f}%")

In [ ]:
# =========================
# SAVE MODEL + TOKENIZER
# =========================
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")
